<a href="https://colab.research.google.com/github/jnrahul92/FastAI_Learn/blob/main/C8_Collaborative_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from fastai.collab import *
from fastai.tabular.all import *

In [3]:
path = untar_data(URLs.ML_100k)

<div><progress max="4924029" value="4931584"></progress> 100.15% [4931584/4924029 00:00&lt;00:00]</div>

In [4]:
ratings = pd.read_csv(path/'u.data', delimiter='\t', header=None, names=['user', 'movie', 'rating', 'timestamp'])
ratings.head()

,user,movie,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


In [5]:
rise_skywalker = np.array([0.98, 0.9, -0.9])

In [6]:
user1 = np.array([0.9, 0.8, -0.6])

In [7]:
(user1*rise_skywalker).sum()

np.float64(2.1420000000000003)

In [8]:
casablanca = np.array([-0.99, -0.3, 0.8])

In [9]:
(user1*casablanca).sum()

np.float64(-1.611)

In [10]:
movies = pd.read_csv(path/'u.item',  delimiter='|', encoding='latin-1', usecols=range(5), names=['movie', 'title', 'release', 'video release', 'imdb url'])
movies.head()

,movie,title,release,video release,imdb url
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%20(1995)
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(1995)
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%20(1995)
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%20(1995)
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995)


In [11]:
ratings = ratings.merge(movies)
ratings.head()

,user,movie,rating,timestamp,title,release,video release,imdb url
0,196,242,3,881250949,Kolya (1996),24-Jan-1997,NaN,http://us.imdb.com/M/title-exact?Kolya%20(1996)
1,186,302,3,891717742,L.A. Confidential (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?L%2EA%2E+Confidential+(1997)
2,22,377,1,878887116,Heavyweights (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Heavyweights%20(1994)
3,244,51,2,880606923,Legends of the Fall (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Legends%20of%20the%20Fall%20(1994)
4,166,346,1,886397596,Jackie Brown (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?imdb-title-119396


In [12]:
dls = CollabDataLoaders.from_df(ratings, item_name='title', bs=64)
dls.show_batch()

,user,title,rating
0,178,"Sound of Music, The (1965)",4
1,826,Fantasia (1940),3
2,576,Everyone Says I Love You (1996),3
3,26,Mystery Science Theater 3000: The Movie (1996),3
4,455,Fried Green Tomatoes (1991),4
5,537,Diva (1981),3
6,393,Cliffhanger (1993),3
7,24,Bound (1996),3
8,617,Army of Darkness (1993),1
9,125,"Silence of the Lambs, The (1991)",5


In [13]:
n_users = len(dls.classes['user'])
n_movies = len(dls.classes['title'])
n_factors = 5

In [14]:
user_factors = torch.randn(n_users, n_factors)
movie_factors = torch.randn(n_movies, n_factors)
user_factors.shape, movie_factors.shape

(torch.Size([944, 5]), torch.Size([1665, 5]))

In [15]:
class DotProduct(Module):
  def __init__(self, n_users, n_movies, n_factors):
    self.user_factors = Embedding(n_users, n_factors)
    self.movie_factors = Embedding(n_movies, n_factors)

  def forward(self, x):
    users = self.user_factors(x[:,0])
    movies = self.movie_factors(x[:,1])
    return (users * movies).sum(dim = 1)

In [16]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())

In [17]:
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,1.325047,1.338394,00:07
1,1.048074,1.113242,00:07
2,0.897502,0.998516,00:06
3,0.785739,0.910101,00:07
4,0.751446,0.886033,00:06


In [18]:
class DotProduct(Module):
  def __init__(self, n_users, n_movies, n_factors, y_range = (0.5,5.5)):
    self.user_factors = Embedding(n_users, n_factors)
    self.movie_factors = Embedding(n_movies, n_factors)
    self.y_range = y_range

  def forward(self, x):
    users = self.user_factors(x[:,0])
    movies = self.movie_factors(x[:,1])
    return sigmoid_range((users * movies).sum(dim = 1),*self.y_range)

In [19]:
model = DotProduct(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.958647,0.993740,00:07
1,0.605816,0.941620,00:08
2,0.445083,0.960585,00:07
3,0.332111,0.969047,00:07
4,0.323691,0.968924,00:06


In [20]:
class DotProductBias(Module):
  def __init__(self, n_users, n_movies, n_factors, y_range = (0.5,5.5)):
    self.user_factors = Embedding(n_users, n_factors)
    self.user_bias = Embedding(n_users, 1)
    self.movie_factors = Embedding(n_movies, n_factors)
    self.movie_bias = Embedding(n_movies, 1)
    self.y_range = y_range

  def forward(self, x):
    users = self.user_factors(x[:,0])
    movies = self.movie_factors(x[:,1])
    res = (users * movies).sum(dim = 1, keepdim = True)
    res += self.user_bias(x[:,0]) + self.movie_bias(x[:,1])
    return sigmoid_range(res,*self.y_range)

In [21]:
model = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3)

epoch,train_loss,valid_loss,time
0,0.848930,0.933637,00:08
1,0.572976,0.916287,00:08
2,0.399092,0.943160,00:07
3,0.313963,0.953959,00:08
4,0.289310,0.955959,00:07


In [22]:
model = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.853044,0.940095,00:07
1,0.653139,0.887867,00:07
2,0.528025,0.873233,00:07
3,0.433806,0.859722,00:08
4,0.433229,0.855636,00:08


In [23]:
def create_params(size):
  return nn.Parameter(torch.zeros(*size).normal_(0, 0.01))

In [28]:
class DotProductBias(Module):
  def __init__(self, n_users, n_movies, n_factors, y_range = (0.5,5.5)):
    self.user_factors = create_params([n_users, n_factors])
    self.user_bias = create_params([n_users])
    self.movie_factors = create_params([n_movies, n_factors])
    self.movie_bias = create_params([n_movies])
    self.y_range = y_range

  def forward(self, x):
    users = self.user_factors[x[:,0]]
    movies = self.movie_factors[x[:,1]]
    res = (users * movies).sum(dim = 1)
    res += self.user_bias[x[:,0]] + self.movie_bias[x[:,1]]
    return sigmoid_range(res,*self.y_range)

In [29]:
model = DotProductBias(n_users, n_movies, 50)
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.879097,0.944574,00:08
1,0.648569,0.882486,00:08
2,0.523667,0.868608,00:07
3,0.447138,0.856993,00:08
4,0.402680,0.852866,00:10


In [30]:
movie_bias = learn.model.movie_bias.squeeze()

In [31]:
idxs = movie_bias.argsort()[:5]

In [32]:
[dls.classes['title'][i] for i in idxs]

['Children of the Corn: The Gathering (1996)',
 'Mortal Kombat: Annihilation (1997)',
 'Island of Dr. Moreau, The (1996)',
 'Speed 2: Cruise Control (1997)',
 'Vampire in Brooklyn (1995)']

In [33]:
idxs = movie_bias.argsort(descending=True)[:5]
[dls.classes['title'][i] for i in idxs]

["Schindler's List (1993)",
 'Silence of the Lambs, The (1991)',
 'Titanic (1997)',
 'Shawshank Redemption, The (1994)',
 'Star Wars (1977)']

In [34]:
learn = collab_learner(dls, n_factors=50, y_range=(0, 5.5))
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.901855,0.950038,00:08
1,0.684317,0.889958,00:07
2,0.517173,0.872811,00:08
3,0.460999,0.859981,00:07
4,0.428444,0.855249,00:08


In [35]:
learn.model

EmbeddingDotBias(
  (u_weight): Embedding(944, 50)
  (i_weight): Embedding(1665, 50)
  (u_bias): Embedding(944, 1)
  (i_bias): Embedding(1665, 1)
)

In [36]:
movie_bias = learn.model.i_bias.weight.squeeze()
idxs = movie_bias.argsort(descending=True)[:5]
[dls.classes['title'][i] for i in idxs]

['Shawshank Redemption, The (1994)',
 'Titanic (1997)',
 'Silence of the Lambs, The (1991)',
 'Close Shave, A (1995)',
 'L.A. Confidential (1997)']

In [39]:
movie_factors = learn.model.i_weight.weight
idx = dls.classes['title'].o2i['Silence of the Lambs, The (1991)']
distances = nn.CosineSimilarity(dim=1)(movie_factors, movie_factors[idx][None])
idx = distances.argsort(descending=True)[1:5]
dls.classes['title'][idx]

['Glory (1989)', 'Shawshank Redemption, The (1994)', 'Hour of the Pig, The (1993)', 'Casablanca (1942)']

In [40]:
embs = get_emb_sz(dls)
embs

[(944, 74), (1665, 102)]

In [41]:
class CollabNN(Module):
  def __init__(self, user_sz, item_sz, y_range = (0.5,5.5), n_act = 100):
    self.user_factors = Embedding(*user_sz)
    self.item_factors = Embedding(*item_sz)
    self.layers = nn.Sequential(
        nn.Linear(user_sz[1] + item_sz[1], n_act),
        nn.ReLU(),
        nn.Linear(n_act, 1)
    )
    self.y_range = y_range

  def forward(self, x):
    embs = self.user_factors(x[:,0]), self.item_factors(x[:,1])
    x = self.layers(torch.cat(embs, dim = 1))
    return sigmoid_range(x, *self.y_range)

In [42]:
model = CollabNN(*embs)

In [43]:
learn = Learner(dls, model, loss_func=MSELossFlat())
learn.fit_one_cycle(5, 5e-3, wd=0.01)

epoch,train_loss,valid_loss,time
0,0.911467,0.948550,00:10
1,0.821623,0.914928,00:08
2,0.805378,0.888130,00:08
3,0.789052,0.875574,00:09
4,0.752344,0.867059,00:08


In [44]:
learn = collab_learner(dls, use_nn=True, y_range=(0, 5.5), layers=[100,50])
learn.fit_one_cycle(5, 5e-3, wd=0.1)

epoch,train_loss,valid_loss,time
0,0.940117,0.983705,00:11
1,0.899433,0.923834,00:10
2,0.819529,0.896077,00:10
3,0.754200,0.884265,00:10
4,0.748540,0.875409,00:10
